# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR^2 Dataset `vq4a-28xa`](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the [mlcroissant](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed in the environment
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Get the metadata object
metadata = dataset.metadata
print("Dataset Title:", metadata.name)
print("Description:", metadata.description)


## 2. Data Overview

Review available record sets, their `@id`s, fields, and columns. All exploration will reference the `@id` fields for consistency.

In [ ]:
# List all available record sets, fields, and columns by @id
print("Available Record Sets:")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"- @id: {rs['@id']} | name: {rs['name']}")

print("\nFor each record set, fields and columns:")
for rs in record_sets:
    print(f"\nRecord Set: {rs['name']} (@id: {rs['@id']})")
    # Fields
    if 'fields' in rs:
        print("Fields:")
        for f in rs['fields']:
            print(f"  - @id: {f['@id']} | name: {f['name']} | dataType: {f.get('dataType', '')}")
    # Columns
    if 'columns' in rs:
        print("Columns:")
        for c in rs['columns']:
            print(f"  - @id: {c['@id']} | name: {c['name']} | dataType: {c.get('dataType', '')}")


## 2.1. Inspecting Example Records
You can print example records from any record set using its `@id`.

In [ ]:
# Print up to 2 example records from each record set
for rs in record_sets:
    print(f"\nSample records from record set: {rs['name']} (@id: {rs['@id']})")
    try:
        for i, record in enumerate(dataset.records(record_set=rs['@id'])):
            print(json.dumps(record, indent=2))
            if i == 1:  # only show first 2
                break
    except Exception as e:
        print(f"  Could not load records: {e}")


## 3. Data Extraction

Load data from all available record sets into DataFrames for further analysis. Always reference entities by their `@id`.

In [ ]:
# We'll load all record sets and store as pandas DataFrames in a dictionary keyed by their @id
dataframes = {}
record_set_ids = [rs['@id'] for rs in record_sets]
for rs_id in record_set_ids:
    try:
        df = pd.DataFrame(dataset.records(record_set=rs_id))
        dataframes[rs_id] = df
        print(f"Loaded: {rs_id} | shape: {df.shape}")
    except Exception as e:
        print(f"Failed to load records for {rs_id}: {e}")

# Example: Print columns for the main record set (choose the main data table)
# We select the first record set as an example
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id and main_record_set_id in dataframes:
    print("\nColumns in main record set DataFrame (@id: {}):".format(main_record_set_id))
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No main record set found.")

## 4. Exploratory Data Analysis (EDA)

Apply data processing steps: filter records, normalize numeric fields, group by attributes, etc.

In [ ]:
# ==== Edit this block to select the desired numeric and group fields by their @id ====
record_set_id = main_record_set_id
df = dataframes.get(record_set_id)

# List columns for reference
if df is not None:
    print(f"Available columns in record set {record_set_id}:\n", df.columns.tolist())
    # Try to guess a numeric field
    numeric_candidates = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]  # e.g., '@id-of-numeric-field'
        print(f"Using numeric field: {numeric_field_id}")
    else:
        numeric_field_id = None
        print("No numeric field found!")
    # Try to guess a group-by field (categorical)
    group_candidates = [c for c in df.columns if pd.api.types.is_string_dtype(df[c])]
    group_field_id = group_candidates[0] if group_candidates else None

    threshold = 10
    if numeric_field_id:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df = filtered_df.copy()
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by group_field if available
        if group_field_id is not None and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("No numeric field available for EDA.")
else:
    print(f"No DataFrame loaded for record set {record_set_id}.")

## 5. Visualization

Visualize distributions & relationships between fields in a record set. Update `numeric_field_id` and `group_field_id` as needed from the EDA output above.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group (if available)
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,6))
        # Show at most 10 groups for readability
        top_groups = df[group_field_id].value_counts().head(10).index
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df[df[group_field_id].isin(top_groups)])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
else:
    print("No suitable data available for plotting.")

## 6. Conclusion

- This notebook demonstrated how to load, inspect, process, and visualize data from the [FAIR^2 mass spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using `mlcroissant`.
- For any operation, we referenced all entities (record sets, fields, columns) by their `@id`.
- You can extend these analyses by selecting specific record sets (using their `@id`), filtering by other variables, or integrating into your own processing pipeline.

For more advanced analysis or custom field extraction, consult the official [mlcroissant documentation](https://github.com/mlcommons/croissant) and your dataset's Croissant schema.